## Reconciliation - the raw CNAF rows, mapped, carrying their codes

Clone of `clean_cnaf_1_before_qf_batch.ipynb` that drops **no column** until the very end,
merged onto the dated `*-with-codes.csv` files `generate_new_codes.ipynb` has already written.

- Replay phase 1 from the raw CNAF file, keeping every raw and intermediate column
- Shape the allocataire-* pivot columns the same way the qf-batch input does, for every
  row (not just the ARS ones qf-batch.ts is actually called for) - matricule,
  code_organisme, nom_naissance, nom_usage, prenom, date_naissance, genre,
  code_insee_naissance, code_pays_naissance: everything a quotient_familial API call needs
- Load the coded rows of both CNAF routes (quotient familial, and AAH/AEEH)
- Merge them on what the export kept, giving every `id_psp` back its matricule, its
  address lines and its ORIGINESELECTION
- Keep the `beneficiaires` columns of the codes files, `allocataire` / `adresse_allocataire`
  JSON included, and add the allocataire identity the export had dropped as plain columns
  named after lamp01's `beneficiaire_cnaf_extra_field` table (see `reconcile.EXTRA_FIELD_COLUMNS`)
- Write one row per code to `CNAF_RECONCILED_PATHFILE_2026`, which `lamp01/inject_csv.sh`
  loads into both tables at once

Read-only with respect to the campaign: it writes no qf-batch input, does not touch the
phase-1 parquet and draws no code. Running it twice changes nothing.

## What differs from clean_cnaf_1_before_qf_batch.ipynb
- `cnaf.drop_raw_address_columns` is never called: NOMCOMPLET and ADRLIG1..6 stay available
  for the whole replay, in case a later step needs them, and are only left out of the output
  by `reconcile.select_output_columns`
- `partners.filter_rows_missing_required_fields` becomes
  `reconcile.filter_rows_missing_required_fields`: the same row filter, without the
  all-null column drop bundled into it
- `reconcile.prepare_qf_identity_columns` runs on every row, not only the ARS-origin
  allocataires `partners.select_qf_route_allocataires` would keep for the qf-batch input
- the qf-batch input cell is gone, so the file qf-batch.ts reads cannot be overwritten

## ⚠️ allocataire-date_naissance / -genre / -code_insee_naissance / -code_pays_naissance
CNAF only fills DTNAIDOS/SEXDOS/COMMUNENAIDOS/PAYSNAIDOS for ARS-origin rows in the raw
file itself (see the CNAF_COLUMN_MAPPING comment in clean_cnaf_lib.py) - an AAH or AEEH
row has them blank in the export CNAF sends us, confirmed against CDBSP1O1N.csv. These
columns here are read straight off that raw file with no further transformation beyond
the same shaping qf-batch.ts's input gets, so a blank value on an AAH/AEEH row is the raw
file's own content, not something this notebook could recover by reading it differently.
`allocataire-nom_naissance` (NOMNAIDOS) is the exception - CNAF fills it for every route.

## Prerequisite
`generate_new_codes.ipynb` must have run on `CNAF` and on `CNAF_AAH_AEEH`, off the very
same raw CNAF file as the one replayed here.

In [ ]:
import csv
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# partners_lib imports utils.data_utils, which lives at the data/ root: make that root
# importable first, since this notebook runs from its own directory.
try:
    import utils.data_utils  # noqa: F401
except ModuleNotFoundError:
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "utils" / "data_utils.py").exists():
            sys.path.append(str(parent))
            break

# partners_lib itself sits one level up, in partners/, next to the other partner folders.
partners_root = str(Path.cwd().parent)
if partners_root not in sys.path:
    sys.path.append(partners_root)

from utils.data_utils import get_current_date_for_file_name

import partners_lib as partners
import clean_cnaf_lib as cnaf
import reconcile_cnaf_lib as reconcile

load_dotenv()

cnaf_input_filepath = os.environ['CNAF_PATHFILE_2026']

# INSEE COG countries & territories, used to turn CNAF's PAYSNAIDOS label into a COG code
# for allocataire-code_pays_naissance. Defaults to the copy shipped in partners/, shared
# with the other partners - same file clean_cnaf_1_before_qf_batch.ipynb reads.
cog_pays_input_filepath = os.environ.get(
    'COG_PAYS_PATHFILE_2026', str(partners_root / Path('v_pays_territoire_2026.csv')))

# The dated files generate_new_codes.ipynb wrote, one per CNAF route ('CNAF' and
# 'CNAF_AAH_AEEH'). Comma-separated in CNAF_WITH_CODES_PATHFILES_2026; defaults to every
# *cnaf*-with-codes.csv file. On the processing machine every CNAF csv - raw exports and
# the coded files alike - lives directly next to this notebook, regardless of where the
# DB_CNAF_EXPORT_2026* env vars point, so that is where the glob looks (Path.cwd(), not
# an env var's parent directory).
codes_directory = Path.cwd()

codes_filepaths_from_env = os.environ.get('CNAF_WITH_CODES_PATHFILES_2026')
codes_filepaths = (
    [Path(filepath.strip()) for filepath in codes_filepaths_from_env.split(',')]
    if codes_filepaths_from_env
    else sorted(codes_directory.glob('*cnaf*-with-codes.csv')))

if not codes_filepaths:
    raise FileNotFoundError(
        f"No '*cnaf*-with-codes.csv' file found in {codes_directory}. "
        "generate_new_codes.ipynb must have run on both 'CNAF' and 'CNAF_AAH_AEEH' "
        "for this raw CNAF file before this notebook can reconcile them."
    )

# Named so it can never be picked up by the glob above on a later run.
reconciled_filepath = os.environ.get(
    'CNAF_RECONCILED_PATHFILE_2026',
    str(codes_directory / get_current_date_for_file_name('cnaf-codes-with-raw-columns.csv')))

print(f"raw CNAF: {cnaf_input_filepath}")
print(f"COG pays: {cog_pays_input_filepath}")
for codes_filepath in codes_filepaths:
    print(f"codes:    {codes_filepath}")
print(f"output:   {reconciled_filepath}")

In [ ]:
# CNAF - column names are supplied positionally, not read from the file's own header row
# (see clean_cnaf_lib.read_raw_cnaf_csv: that header row has shipped truncated before).
cnaf_df = cnaf.read_raw_cnaf_csv(cnaf_input_filepath)

print(f"{len(cnaf_df)} raw row(s) read from {cnaf_input_filepath}")

In [ ]:
# delete last row (it is not a valid row) & clean white spaces within all columns
cnaf_df = cnaf.clean_raw_cnaf(cnaf_df)

In [ ]:
# Explode postal code & commune from initial column containing both
cnaf_df = cnaf.split_postal_code_and_commune(cnaf_df)

In [ ]:
# Clean extra white spaces
cnaf_df = cnaf.normalize_full_name_spacing(cnaf_df)

In [ ]:
# map CNAF columns to the PSP schema (see cnaf.CNAF_COLUMN_MAPPING)
df_psp_mapped_cnaf = cnaf.map_cnaf_columns(cnaf_df)

In [ ]:
# Allocataire missing phone number
df_psp_mapped_cnaf = partners.clear_placeholder_phone_numbers(df_psp_mapped_cnaf)

# Allocataire's qualite
df_psp_mapped_cnaf = partners.normalize_allocataire_qualite(df_psp_mapped_cnaf)

# Additionnal address details & allocataire's street address
df_psp_mapped_cnaf = cnaf.build_allocataire_address_fields(df_psp_mapped_cnaf)

# Organism & situation - CNAF now flags the category itself, no more DOB+name guessing
df_psp_mapped_cnaf = cnaf.set_organisme_and_situation(df_psp_mapped_cnaf)

In [ ]:
# Format date_naissance to datetime python object for processing
df_psp_mapped_cnaf = partners.parse_beneficiary_birthdate(df_psp_mapped_cnaf)

In [ ]:
# Shape the allocataire-* pivot columns exactly like the qf-batch input does - matricule,
# code_organisme, nom_naissance, prenom and code_insee_naissance are already there from
# map_cnaf_columns; this adds nom_usage and code_pays_naissance, and reformats genre /
# date_naissance to the same shape a quotient_familial API call expects. Unlike
# clean_cnaf_1_before_qf_batch.ipynb, this runs on every row, not only the ARS-origin ones -
# see the ⚠️ note above for why an AAH/AEEH row still ends up with these columns blank.
df_cog_pays = pd.read_csv(cog_pays_input_filepath, dtype=str, keep_default_na=False)
cog_by_country_label = partners.build_country_cog_lookup(df_cog_pays)

df_psp_mapped_cnaf, unmapped_labels, born_abroad_count = reconcile.prepare_qf_identity_columns(
    df_psp_mapped_cnaf, cog_by_country_label)

if unmapped_labels:
    print(f"{len(unmapped_labels)} PAYSNAIDOS label(s) without a COG match: {unmapped_labels}")
print(f"{born_abroad_count} allocataire(s) born outside France: "
      "allocataire-code_insee_naissance cleared")

## ⏸ No qf-batch input is written here
`clean_cnaf_1_before_qf_batch.ipynb` writes `QF_BATCH_INPUT_PATHFILE_2026` at this point.
That cell is deliberately absent: qf-batch.ts has already run on that file and this
notebook must not overwrite it. Nothing below needs the verdict either - it reaches these
rows through the codes files, which only exist for beneficiaries a route already selected.

In [ ]:
# clean_cnaf_1 calls cnaf.drop_raw_address_columns here, dropping NOMCOMPLET and ADRLIG1..6
# now that they have been exploded into the adresse_allocataire-* columns. This notebook
# keeps them for the whole replay - reconcile.select_output_columns only leaves them out of
# the file it writes, where the adresse_allocataire JSON says the same thing in structured form.
assert set(cnaf.RAW_ADDRESS_COLUMNS_TO_DROP) <= set(df_psp_mapped_cnaf.columns)

In [ ]:
# remove rows with missing necessary values (if one of those value are missing we cannot
# generate a code). Unlike phase 1 this keeps the columns left entirely null.
df_valid = reconcile.filter_rows_missing_required_fields(df_psp_mapped_cnaf)

print(f"{len(df_psp_mapped_cnaf) - len(df_valid)} row(s) removed for a missing "
      f"{partners.NECESSARY_COLUMNS}")

In [ ]:
# Upper case these columns for the merge
df_valid = partners.normalize_identity_casing(df_valid)

In [ ]:
# lower case on emails on all
df_valid = partners.normalize_email_casing(df_valid)

In [ ]:
# Preliminary filter, ahead of the precise QF/AAH/AEEH windows applied by phase 2:
# 1996-01-01 is the oldest birthdate any of the 3 routes can accept (AAH's lower bound,
# partners.AAH_DOB_MIN).
df_valid_after = partners.filter_within_eligibility_floor(df_valid)

print(f"{len(df_valid) - len(df_valid_after)} rows removed because they are outside all eligibility windows")

In [ ]:
# add missing 0 to phone numbers, and set '0' phone values to None
df_valid_after = partners.fix_phone_number_formatting(df_valid_after)

In [ ]:
# set Nan values for not existing courriel
df_valid_after = partners.clear_blank_email(df_valid_after)

In [ ]:
# add 4h on all birthdates
df_valid_after = partners.shift_birthdate_by_hours(df_valid_after)

In [ ]:
# remove duplicate beneficiaries (see partners.DEDUPLICATION_KEY_COLUMNS)
df_valid_no_duplicate, duplicate_count = partners.drop_duplicate_beneficiaries(df_valid_after)

print(f"{duplicate_count} duplicate rows were removed")

In [ ]:
# map allocataire json
df_valid_no_duplicate = partners.add_allocataire_json_column(df_valid_no_duplicate)

In [ ]:
# map adresse_allocataire json
df_valid_no_duplicate = partners.add_adresse_allocataire_json_column(df_valid_no_duplicate)

## 🔗 Reconciliation
The frame above is the phase-1 parquet plus every column phase 1 dropped. The codes files
carry the 8 columns the export kept, which are enough to key one onto the other: they are
`partners.DEDUPLICATION_KEY_COLUMNS` with the allocataire-* half folded into its JSON
column (see `reconcile.MERGE_KEY_COLUMNS`).

In [ ]:
# The key is textual on both sides: the export cast date_naissance to string right before
# writing, and the codes files are read back as text.
df_full = reconcile.format_date_naissance_as_exported(df_valid_no_duplicate)

df_full, collision_count = reconcile.drop_merge_key_collisions(df_full)
print(f"{collision_count} row(s) dropped for sharing a merge key with a row already kept")

In [ ]:
# One frame per route, tagged with the file it came from so a reconciled row says which run
# handed out its code. Read as text, the way generate_codes_lib wrote it.
df_codes = pd.concat([
    pd.read_csv(filepath, sep=';', encoding='utf-8', dtype=str, keep_default_na=False)
      .assign(fichier_codes=filepath.name)
    for filepath in codes_filepaths
], ignore_index=True)

assert df_codes['id_psp'].is_unique

print(f"{len(df_codes)} coded row(s) read from {len(codes_filepaths)} file(s)")
print(df_codes['fichier_codes'].value_counts())

In [ ]:
# Many-to-one and validated as such: the merge can neither add nor lose a code.
df_reconciled, unmatched_index = reconcile.merge_codes_with_full_rows(df_codes, df_full)

print(f"{len(df_reconciled)} reconciled row(s), {len(unmatched_index)} code(s) with no match "
      "in the raw file")

# A non-empty sample here means the raw CNAF file is not the one those codes came from.
df_reconciled.loc[unmatched_index, ['id_psp', 'nom', 'prenom', 'date_naissance',
                                    'situation', 'fichier_codes']].head(20)

In [ ]:
# The other side of the merge, for the record: phase-1 rows no code was ever handed to -
# outside their route's window, or in a household qf-batch put above the threshold.
df_without_code = reconcile.select_rows_without_code(df_full, df_codes)

print(f"{len(df_without_code)} phase-1 row(s) without a code")
print(df_without_code['situation'].value_counts(dropna=False))

## 🗄️ The output: beneficiaires columns, then beneficiaire_cnaf_extra_field columns
One row per code, ready for `lamp01/inject_csv.sh`, which loads it into two tables at once:
- the codes files' own columns, `allocataire` and `adresse_allocataire` JSON included and
  left exactly as the export wrote them, go to `beneficiaires` - minus the placeholders
  listed in `reconcile.RECONCILED_COLUMNS_TO_DROP`
- what the replay recovered on top of the JSON's shared core - the allocataire's birth
  name, birthdate, genre and birthplace, see `reconcile.EXTRA_FIELD_COLUMNS` -
  goes to `beneficiaire_cnaf_extra_field`, keyed on `id_psp`. The usage name is not among
  them: CNAF defaults it to the same RESPDOS value already serialized as the JSON's "nom" key,
  so a second copy would only duplicate that field.

Every raw or staging column the replay carried (NOMCOMPLET, ADRLIG1..6, the allocataire-*
core the JSON already holds, situation_origine) is left out.

The codes it holds are the codes files' own: it **replaces** them in `beneficiaires`, it
does not come on top of them - `beneficiaires_id_psp_unique` rejects the whole injection
if any of these `id_psp` is already there.

```bash
./lamp01/inject_csv.sh <CNAF_RECONCILED_PATHFILE_2026>              # integration
./lamp01/inject_csv.sh --env prod <CNAF_RECONCILED_PATHFILE_2026>
```

In [ ]:
df_output = reconcile.select_output_columns(df_reconciled, df_codes)

assert len(df_output) == len(df_codes)
assert df_output['id_psp'].is_unique

In [ ]:
df_output.to_csv(reconciled_filepath, sep=';', index=False, encoding='utf-8',
                 quoting=csv.QUOTE_ALL)

print(f"{len(df_output)} row(s) x {len(df_output.columns)} column(s) written to "
      f"{reconciled_filepath}")
print(list(df_output.columns))